# Synthese:
- bool(6), float64(7), int64(8), str(5) the str are only nominal, i transformed them into hot ones
- Unbalanced class fraud vs unfraud, a very small quantity is fraud ( 1.67%) (no preprocess for it for now)
- NO missing values

In [8]:
#Read the  dataset 
import pandas as pd
path= '/home/paul/.cache/kagglehub/competitions/playground-series-s6e11'
df_train = pd.read_csv(f'{path}/train.csv', index_col=0)
df_test = pd.read_csv(f'{path}/test.csv', index_col=0)



In [9]:
df_train.shape
df_train.info()
df_train.describe()
df_train.isna().sum()
df_train.duplicated().sum()
df_train['is_fraud'].value_counts() 

<class 'pandas.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   amount_usd                    699960 non-null  float64
 1   hours_since_last_txn          699960 non-null  float64
 2   txn_count_last_24h            699960 non-null  int64  
 3   distance_from_home_km         699960 non-null  float64
 4   card_age_months               699960 non-null  int64  
 5   customer_age                  699960 non-null  int64  
 6   account_balance_usd           699960 non-null  float64
 7   cvv_retry_count               699960 non-null  int64  
 8   velocity_score                699960 non-null  float64
 9   time_of_day_hour              699960 non-null  int64  
 10  day_of_week                   699960 non-null  int64  
 11  merchant_risk_score           699960 non-null  float64
 12  prior_disputes                699960 non-null  int64  


is_fraud
0.0    688271
1.0     11689
Name: count, dtype: int64

In [10]:
target = "is_fraud"

y_train = df_train[target].copy()
n_train = len(df_train)

train_features = df_train.drop(columns=[target])
combined = pd.concat([train_features, df_test], ignore_index=True)

combined_encoded = pd.get_dummies(
    combined,
    columns=categorical_features,
    dtype=int,
)

df_encoded_train = combined_encoded.iloc[:n_train].copy()
df_encoded_train[target] = y_train.to_numpy()
df_encoded_test = combined_encoded.iloc[n_train:].copy()

# Restore the original indices (your data was read with index_col=0).
df_encoded_train.index = df_train.index
df_encoded_test.index = df_test.index

In [12]:
#Most correlated features with the target variable (absolute value of correlation coefficient)
correlation_matrix = df_encoded_train.corr()
target_correlation = correlation_matrix['is_fraud'].abs().sort_values(ascending=False)
#display the top 10 most correlated features with the target variable
top_10_features = target_correlation[1:11]
print("Top 10 most correlated features with the target variable:")
print(top_10_features)
#Less correlated features with the target variable ( absolute value of correlation coefficient)
less_correlated_features = target_correlation[-10:]
print("Less correlated features with the target variable:")
print(less_correlated_features)

Top 10 most correlated features with the target variable:
cvv_retry_count                  0.212290
ip_country_mismatch              0.148284
merchant_risk_score              0.125391
auth_method_No Authentication    0.121295
is_foreign_transaction           0.112832
velocity_score                   0.101650
is_new_merchant                  0.101412
billing_shipping_mismatch        0.101115
used_vpn                         0.071235
is_ai_generated_scam_attempt     0.060719
Name: is_fraud, dtype: float64
Less correlated features with the target variable:
account_balance_usd        0.001077
channel_In-App             0.001066
device_type_Mac            0.000709
card_age_months            0.000681
device_type_Tablet         0.000493
channel_POS                0.000486
device_type_Smart Watch    0.000426
device_type_ATM Machine    0.000352
card_type_RuPay            0.000197
time_of_day_hour           0.000050
Name: is_fraud, dtype: float64


In [13]:
#save the encoded dataframe to a new CSV file
df_encoded_train.to_csv(f"{path}/df_encoded_train_hot_ones.csv", index=True)
df_encoded_test.to_csv(f"{path}/df_encoded_test_hot_ones.csv", index=True)
